In [ ]:
import os
import os

folder_path = r"C:\project\political_ner\Final_NER"

try:
    # List all files in the folder
    files = os.listdir(folder_path)

    print("Files in the folder:")
    for file in files:
        print(file)
except FileNotFoundError:
    print(f"The folder at {folder_path} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
import pandas as pd
import os

def create_country_glossary(input_file, output_folder):
    """
    1. Reads a single Excel file (input_file).
    2. Creates 'all_variations' by grouping over 'Names' or 'Party' and collecting unique 'NER'.
    3. Saves only the 'Glossary' sheet with duplicates removed and all required columns included.
    """

    # Read the input file
    df = pd.read_excel(input_file)

    # Ensure required columns exist; create them if missing
    for col in ['NER', 'Names', 'Party', 'NER_country_cleaned', 'EU']:
        if col not in df.columns:
            df[col] = ''

    # Fill NaNs for consistency
    df['NER'] = df['NER'].fillna('')
    df['Names'] = df['Names'].fillna('')
    df['Party'] = df['Party'].fillna('')
    df['NER_country_cleaned'] = df['NER_country_cleaned'].fillna('Unknown')
    df['EU'] = df['EU'].fillna('')

    # Get the country name from the first row (or 'Unknown' if empty)
    country_name = df['NER_country_cleaned'].iloc[0] if not df['NER_country_cleaned'].isnull().all() else 'Unknown'

    # Build separate glossaries
    name_glossary = (df[df['Names'] != '']
                     .groupby('Names')['NER']
                     .apply(lambda x: list(set(x)))
                     .to_dict())

    party_glossary = (df[(df['Names'] == '') & (df['Party'] != '')]
                      .groupby('Party')['NER']
                      .apply(lambda x: list(set(x)))
                      .to_dict())

    # Create a separate glossary DataFrame
    glossary_data = []
    for _, row in df.iterrows():
        glossary_data.append({
            'Type': 'Name' if row['Names'] else 'Party',
            'Entry': row['Names'] if row['Names'] else row['Party'],
            'Variations': ', '.join(name_glossary.get(row['Names'], []) if row['Names'] else party_glossary.get(row['Party'], [])),
            'Party': row['Party'],
            'Country': row['NER_country_cleaned'],
            'EU': row['EU']
        })

    # Convert to DataFrame and remove duplicates
    glossary_df = pd.DataFrame(glossary_data)
    glossary_df = glossary_df.drop_duplicates()  # Remove exact duplicates

    # Determine output file name: "<Country>_Glossary.xlsx"
    output_filename = f"{country_name}_Glossary.xlsx"
    output_path = os.path.join(output_folder, output_filename)

    # Save results to Excel with only one sheet: "Glossary"
    with pd.ExcelWriter(output_path) as writer:
        glossary_df.to_excel(writer, index=False, sheet_name='Glossary')

    print(f"Glossary saved to: {output_path}")


# Example usage:
if __name__ == "__main__":
    input_file_path = r"C:\project\political_ner\FINAL_NER_2025_17_01\NER_Identified\Final_SE_NER_Identified.xlsx"
    output_folder_path = (r"C:\project\political_ner"
                          r"\FINAL_NER_2025_17_01\Glossary")
    
    create_country_glossary(input_file_path, output_folder_path)


In [ ]:
import pandas as pd

file_path = r'C:\project\political_ner\FINAL_NER_2025_17_01\Glossary\bulgaria_Glossary.xlsx'
second_sheet = pd.read_excel(file_path, sheet_name=1)  # Index 1 corresponds to the second sheet
second_sheet.to_excel(file_path, index=False, sheet_name='Glossary')  


In [ ]:
import os
import pandas as pd

# Define the folder path
folder_path = r"C:\project\political_ner\Final_NER"

# Output file name
output_file = os.path.join(folder_path, "final_allcountry.xlsx")

try:
    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter Excel files
    excel_files = [file for file in files if file.endswith('.xlsx')]

    # Merge all Excel files
    merged_df = pd.DataFrame()

    for file in excel_files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)

        # Ensure 'Country' column exists
        if 'Country' in df.columns:
            country = df['Country'].iloc[0] if not df['Country'].isnull().all() else "Unknown"

            # Update the 'NER' and 'Party' columns with the country name
            if 'NER' in df.columns:
                df['NER'] = df['NER'] + f" ({country})"
            if 'Party' in df.columns:
                df['Party'] = df['Party'] + f" ({country})"

        merged_df = pd.concat([merged_df, df], ignore_index=True)

    # Save the merged DataFrame to a new Excel file
    merged_df.to_excel(output_file, index=False)

    print(f"Merged file created: {output_file}")
except FileNotFoundError:
    print(f"The folder at {folder_path} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
import pandas as pd

# File paths
input_file = r'C:\project\political_ner\Final_NER\final_allcountry.xlsx'
output_file = r'C:\project\political_ner\Final_NER\All_withglossary.xlsx'

# Load the Excel file
df = pd.read_excel(input_file)

# Ensure the required columns are present
required_columns = ['NER', 'Names', 'Party']
if not all(col in df.columns for col in required_columns):
    raise ValueError(f"The input file must contain the following columns: {required_columns}")

# Fill NaN values for consistency
df['Names'] = df['Names'].fillna('')
df['Party'] = df['Party'].fillna('')

# Create a glossary for Names and Parties
name_glossary = df[df['Names'] != ''].groupby('Names')['NER'].apply(lambda x: list(set(x))).to_dict()
party_glossary = df[df['Names'] == ''].groupby('Party')['NER'].apply(lambda x: list(set(x))).to_dict()

# Create the all_variations_person and all_variations_party columns
def get_all_variations(row):
    if row['Names']:
        return ', '.join(name_glossary.get(row['Names'], [])), ''
    elif row['Party']:
        return '', ', '.join(party_glossary.get(row['Party'], []))
    else:
        return '', ''

df['all_variations_person'], df['all_variations_party'] = zip(*df.apply(get_all_variations, axis=1))

# Save the updated dataframe to a new Excel file
df.to_excel(output_file, index=False)

print(f"Updated file saved to {output_file}")


In [ ]:
import pandas as pd

# File paths
input_file = r'C:\project\political_ner\Final_NER\final_allcountry.xlsx'
output_file = r'C:\project\political_ner\Final_NER\All_withglossary.xlsx'

# Load the Excel file
df = pd.read_excel(input_file)

# Ensure the required columns are present
required_columns = ['NER', 'Names', 'Party']
if not all(col in df.columns for col in required_columns):
    raise ValueError(f"The input file must contain the following columns: {required_columns}")

# Fill NaN values for consistency
df['Names'] = df['Names'].fillna('')
df['Party'] = df['Party'].fillna('')

# Create a glossary for Names and Parties
name_glossary = df[df['Names'] != ''].groupby('Names')['NER'].apply(lambda x: list(set(x))).to_dict()
party_glossary = df[df['Names'] == ''].groupby('Party')['NER'].apply(lambda x: list(set(x))).to_dict()

# Create a single column for all variations
def get_all_variations(row):
    if row['Names']:
        return ', '.join(name_glossary.get(row['Names'], []))
    elif row['Party']:
        return ', '.join(party_glossary.get(row['Party'], []))
    else:
        return ''

df['all_variations'] = df.apply(get_all_variations, axis=1)

# Create a separate sheet for the glossary
glossary_data = []
for name, variations in name_glossary.items():
    glossary_data.append({'Type': 'Name', 'Entry': name, 'Variations': ', '.join(map(str, variations))})
for party, variations in party_glossary.items():
    glossary_data.append({'Type': 'Party', 'Entry': party, 'Variations': ', '.join(map(str, variations))})

glossary_df = pd.DataFrame(glossary_data)

# Save the updated dataframe and glossary to a new Excel file with multiple sheets
with pd.ExcelWriter(output_file) as writer:
    df.to_excel(writer, index=False, sheet_name='Main')
    glossary_df.to_excel(writer, index=False, sheet_name='Glossary')

print(f"Updated file with glossary saved to {output_file}")


In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Border, Side
import os

# File paths
input_file = r'C:\project\political_ner\Final_NER\final_allcountry.xlsx'
output_file = r'C:\project\political_ner\Final_NER\All_withglossary.xlsx'
output_folder = r'C:\project\political_ner\Glossary'

# Load the Excel file
df = pd.read_excel(input_file)

# Ensure the required columns are present
required_columns = ['NER', 'Names', 'Party', 'Country']
if not all(col in df.columns for col in required_columns):
    raise ValueError(f"The input file must contain the following columns: {required_columns}")

# Fill NaN values for consistency
df['Names'] = df['Names'].fillna('')
df['Party'] = df['Party'].fillna('')
df['Country'] = df['Country'].fillna('Unknown')

# Create a glossary for Names and Parties
name_glossary = df[df['Names'] != ''].groupby('Names')['NER'].apply(lambda x: list(set(x))).to_dict()
party_glossary = df[df['Names'] == ''].groupby('Party')['NER'].apply(lambda x: list(set(x))).to_dict()

# Create a single column for all variations
def get_all_variations(row):
    if row['Names']:
        return ', '.join(name_glossary.get(row['Names'], []))
    elif row['Party']:
        return ', '.join(party_glossary.get(row['Party'], []))
    else:
        return ''

df['all_variations'] = df.apply(get_all_variations, axis=1)

# Create a separate sheet for the glossary
glossary_data = []
for name, variations in name_glossary.items():
    glossary_data.append({'Type': 'Name', 'Entry': name, 'Variations': ', '.join(map(str, variations))})
for party, variations in party_glossary.items():
    glossary_data.append({'Type': 'Party', 'Entry': party, 'Variations': ', '.join(map(str, variations))})

glossary_df = pd.DataFrame(glossary_data)

# Save the updated dataframe and glossary to a new Excel file with formatting
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Main')
    glossary_df.to_excel(writer, index=False, sheet_name='Glossary')

# Load the workbook to apply formatting
wb = load_workbook(output_file)
glossary_sheet = wb['Glossary']

# Apply light blue fill to header
header_fill = PatternFill(start_color='ADD8E6', end_color='ADD8E6', fill_type='solid')
for cell in glossary_sheet[1]:
    cell.fill = header_fill

# Apply borders to the glossary table
thin_border = Border(left=Side(style='thin'),
                     right=Side(style='thin'),
                     top=Side(style='thin'),
                     bottom=Side(style='thin'))
for row in glossary_sheet.iter_rows(min_row=1, max_row=glossary_sheet.max_row,
                                     min_col=1, max_col=glossary_sheet.max_column):
    for cell in row:
        cell.border = thin_border

# Save the workbook with formatting
wb.save(output_file)

# Split the files by country
for country, group in df.groupby('Country'):
    country_file = os.path.join(output_folder, f"NER_{country}_glossary.xlsx")
    with pd.ExcelWriter(country_file, engine='openpyxl') as writer:
        group.to_excel(writer, index=False, sheet_name='Main')
        glossary_df.to_excel(writer, index=False, sheet_name='Glossary')
        
        # Apply formatting to the glossary sheet
        country_wb = writer.book
        glossary_sheet = country_wb['Glossary']
        for cell in glossary_sheet[1]:
            cell.fill = header_fill
        for row in glossary_sheet.iter_rows(min_row=1, max_row=glossary_sheet.max_row,
                                             min_col=1, max_col=glossary_sheet.max_column):
            for cell in row:
                cell.border = thin_border

print(f"Updated file saved to {output_file} and split files by country saved to {output_folder}")


In [ ]:
df.columns